# S3.2 — Reading Data — CSV, JSON, Parquet, Delta, ORC, Avro
**Date completed:** September 2026  
**Status:** In Progress  
**Key concept:** Right format = right performance

In [0]:
# ============================================================
# Cell 2 — Explore available sample datasets
#
# Goal:
# 1. Find datasets with CSV and JSON formats
# 2. Identify paths for reading demonstrations
# ============================================================

print("=== FINDING CSV AND JSON DATASETS ===")
print()

# ------------------------------------------------------------
# Step 1 — Check retail data (closest to our Olist project)
# ------------------------------------------------------------

print("--- Retail dataset ---")
display(dbutils.fs.ls("dbfs:/databricks-datasets/retail-org/"))

In [0]:
# ============================================================
# Cell 3 — Check file formats in retail dataset
#
# Goal:
# 1. Find CSV, JSON, Parquet files
# 2. Identify paths for each format
# ============================================================

print("=== CHECKING FILE FORMATS ===")
print()

# ------------------------------------------------------------
# Step 1 — Check sales_orders (likely CSV or JSON)
# ------------------------------------------------------------

print("--- sales_orders ---")
display(dbutils.fs.ls("dbfs:/databricks-datasets/retail-org/sales_orders/"))
print()

# ------------------------------------------------------------
# Step 2 — Check customers
# ------------------------------------------------------------
print("--- customers ---")
display(dbutils.fs.ls("dbfs:/databricks-datasets/retail-org/customers/"))

In [0]:
# ============================================================
# Cell 4 — Check customers and products format
# ============================================================

print("--- customers ---")
display(dbutils.fs.ls("dbfs:/databricks-datasets/retail-org/customers/"))
print()

print("--- products ---")
display(dbutils.fs.ls("dbfs:/databricks-datasets/retail-org/products/"))

In [0]:
# ============================================================
# Cell 5 — Check for Parquet format
# ============================================================

print("--- purchase_orders ---")
display(dbutils.fs.ls("dbfs:/databricks-datasets/retail-org/purchase_orders/"))

In [0]:
# ============================================================
# Cell 6 — Find Parquet format in sample datasets
# ============================================================

print("--- checking for parquet ---")
display(dbutils.fs.ls("dbfs:/databricks-datasets/retail-org/active_promotions/"))

In [0]:
# ============================================================
# Cell 7 — Reading CSV files
#
# Goal:
# 1. Read CSV with all important options
# 2. Understand each option and why it matters
# 3. Check schema and row count
# ============================================================

print("=== READING CSV ===")
print()

# ------------------------------------------------------------
# Step 1 — Basic CSV read (bad practice — avoid in production)
# ------------------------------------------------------------

print("--- Basic read (no options) ---")
df_basic = spark.read.csv(
    "dbfs:/databricks-datasets/retail-org/customers/customers.csv"
)

df_basic.show(3)
df_basic.printSchema()
print()

# ------------------------------------------------------------
# Step 2 — Production CSV read (with all important options)
# ------------------------------------------------------------

df_customers = (
    spark.read \
        .option("header", "true")  \
        .option("inferschema", "true")  \
        .option("sep", ",") \
        .option("nullValue", "NA") \
        .option("mode", "PERMISSIVE") \
        .csv("dbfs:/databricks-datasets/retail-org/customers/customers.csv")
)

print(f"Row count : {df_customers.count()}")
print(f"Columns   : {len(df_customers.columns)}")
print(f"Column names: {df_customers.columns}")
print()
df_customers.printSchema()
df_customers.show(5)

In [0]:
# ============================================================
# Cell 8 — Reading JSON files
#
# Goal:
# 1. Read JSON with important options
# 2. Compare with CSV reading
# 3. Handle nested JSON structure
# ===========================================================

print("=== READING JSON ===")
print()

# ------------------------------------------------------------
# Step 1 — Read sales orders JSON
# ------------------------------------------------------------

df_sales =  spark.read  \
    .option("inferSchema", "true") \
    .option("mode", "PERMISSIVE") \
    .json("dbfs:/databricks-datasets/retail-org/sales_orders/")

print(f"Row count : {df_sales.count()}")
print(f"Columns   : {len(df_sales.columns)}")
print()

# ------------------------------------------------------------
# Step 2 — Check schema — JSON often has nested structure
# ------------------------------------------------------------
print("--- Schema ---")
df_sales.printSchema()
print()

# ------------------------------------------------------------
# Step 3 — Show sample rows
# ------------------------------------------------------------
print("--- Sample rows ---")
df_sales.show(3, truncate=50)

In [0]:
# ============================================================
# Cell 9 — Reading Parquet files
#
# Goal:
# 1. Read Parquet — notice how much simpler it is
# 2. Schema is embedded — no inferSchema needed
# 3. Compare with CSV and JSON reading
# ============================================================

print("=== READING PARQUET ===")
print()

# ------------------------------------------------------------
# Step 1 — Read Parquet file
# No options needed — schema is embedded in the file
# ------------------------------------------------------------
df_promotions = spark.read \
    .parquet("dbfs:/databricks-datasets/retail-org/active_promotions/active_promotions.parquet")

print(f"Row count : {df_promotions.count()}")
print(f"Columns   : {len(df_promotions.columns)}")
print()

# ------------------------------------------------------------
# Step 2 — Schema comes from file — no inferSchema needed
# ------------------------------------------------------------
print("--- Schema (embedded in file — no inferSchema needed) ---")
df_promotions.printSchema()
print()

# ------------------------------------------------------------
# Step 3 — Show sample rows
# ------------------------------------------------------------
print("--- Sample rows ---")
df_promotions.show(5)
print()

# ------------------------------------------------------------
# Step 4 — Key comparison
# ------------------------------------------------------------
print("=== FORMAT COMPARISON ===")
print(f"CSV    → header + inferSchema needed → slower")
print(f"JSON   → inferSchema needed → nested complexity")
print(f"Parquet→ schema embedded → fastest → use in Silver/Gold")

# S3.2 — Reading Data: Key Takeaways

## 1. Format Comparison

| Format | Schema Handling | Storage | Common Use Case |
|---|---|---|---|
| CSV | Explicit or inferred | Text, row-oriented | Source exports and ingestion |
| JSON | Explicit or inferred automatically | Semi-structured text | APIs, events, nested data |
| Parquet | Schema stored in file metadata | Columnar | Analytical data |
| Delta | Schema and metadata managed through Delta log; Parquet data files | Transactional table format | Databricks lakehouse tables |
| ORC | Schema stored in file metadata | Columnar | Hive ecosystem and analytics |
| Avro | Schema-based binary format | Row-oriented | Event exchange and serialization |

Performance depends on data volume, compression, file layout,
query patterns and execution environment.

Delta is the default table format in Databricks, but no format is
universally the fastest for every workload.

---

## 2. Key CSV Reader Options

| Option | Purpose |
|---|---|
| header=true | Use the first row as column names |
| inferSchema=true | Infer column data types |
| sep="," | Specify the delimiter (comma is the default) |
| nullValue="NA" | Interpret the specified value as null |
| mode="PERMISSIVE" | Handle malformed records permissively |

Example:

```python
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .option("nullValue", "NA")
    .option("mode", "PERMISSIVE")
    .csv(file_path)
)
```

For production workloads, prefer an explicit schema when a reliable
source-data contract is available.

---

## 3. The inferSchema Problem

For CSV, inferSchema=true requires an additional pass over the input
to determine data types.

Potential problems:
- Additional schema-discovery cost.
- Unexpected column types.
- IDs with leading zeros may be interpreted incorrectly.
- Schema inference can produce unexpected results with inconsistent data.

Solution:

Use an explicit StructType schema when the expected schema is known.

This avoids the inference step and provides predictable data types,
provided the supplied schema is correct.

Detailed explicit-schema implementation: S3.3.

---

## 4. Reading JSON

Basic syntax:

```python
df_json = spark.read.json(json_path)
```

Spark automatically infers the JSON schema when one is not supplied.

Important JSON reading options:

- multiLine=true: For appropriate multi-line JSON files.
- mode=PERMISSIVE: Permissive malformed-record handling.
- samplingRatio: Controls the fraction of records used for schema inference.

Standard JSON Lines files contain one JSON object per line.

### Nested JSON

Enterprise JSON may contain:

- Structs inside structs.
- Arrays of structs.
- Arrays containing other arrays.
- Optional or missing fields.

Illustrative RetailPulse examples:

- clicked_items
- ordered_products
- promotion_info

Useful functions include:

- explode()
- explode_outer()
- col()
- select()
- from_json()

These will be explored in S3.5.

---

## 5. Reading Parquet

Syntax:

```python
df_parquet = spark.read.parquet(parquet_path)
```

Parquet stores schema information in its file metadata.

Benefits:
- Columnar storage.
- Efficient compression and encoding.
- Column pruning.
- Predicate pushdown where supported.
- Suitable for analytical processing.

Example:

If a Parquet file contains 50 columns but our query requires
only 3, Spark can often avoid reading unnecessary column data.

---

## 6. Reading Delta

Syntax:

```python
df_delta = spark.read.format("delta").load(delta_path)
```

Or read a registered table:

```python
df_delta = spark.table("catalog.schema.table_name")
```

Delta Lake uses Parquet data files together with a transaction log.

Important capabilities:

- ACID transactions.
- Schema enforcement.
- Controlled schema evolution.
- Table history and time travel.
- Support for reliable batch and streaming processing.

Delta is Databricks' default table format.

---

## 7. ORC and Avro

Basic reading syntax:

```python
# ORC
df_orc = spark.read.orc(orc_path)

# Avro
df_avro = spark.read.format("avro").load(avro_path)
```

ORC is another columnar analytical file format.

Avro is a schema-based binary format commonly used for data exchange
and event-oriented workloads.

Practical execution will be completed separately.

---

## 8. Enterprise Reading Strategy

For RetailPulse:

Source CSV / JSON / Other Formats
              |
              v
       Read Source Data
              |
              v
       Validate Schema
              |
              v
      Handle Invalid Records
              |
              v
        Bronze Delta
              |
              v
    Silver Transformations
              |
              v
         Gold Delta

Important considerations:

1. Understand the source format before reading.
2. Use a reliable schema contract when available.
3. Handle malformed records explicitly.
4. Preserve source data where required for auditing.
5. Avoid unnecessary schema inference in recurring pipelines.
6. Validate row counts, data types and required fields.
7. Use Delta for managed lakehouse processing where appropriate.
8. Select only required columns when possible.

---

## 9. Practical Completion Status

| Topic | Status |
|---|---|
| CSV reading | Covered in notebook |
| JSON reading | Covered in notebook |
| Parquet reading | Covered in notebook |
| Delta reading | Practical pending |
| ORC reading | Practical pending |
| Avro reading | Practical pending |
| Nested JSON flattening | Planned for S3.5 |
| Explicit schemas | Planned for S3.3 |

---

## 10. Interview Takeaways

Q: Why is Parquet generally more efficient than CSV for analytical queries?

Because Parquet stores data in columns, supports compression and
can avoid reading unnecessary columns.

Q: Why should we avoid relying blindly on inferSchema?

Because inference adds discovery overhead and may produce data types
that differ from the intended source contract.

Q: What is the difference between Parquet and Delta?

Parquet is a columnar file format.

Delta extends Parquet data files with a transaction log and
table-management capabilities such as ACID transactions.

Q: Which format should we use for source ingestion?

Use the appropriate reader for the actual source format.
Validate the schema and establish a suitable target storage format.

---

## Final Learning Principle

Reading data is not just loading a file into a DataFrame.

A Data Engineer must understand the source format, schema, data quality,
performance characteristics and target storage requirements.

**Choose the correct format for the workload, not simply the format
with the fastest reputation.**